# Example of XML shredding (normalization) with file using the Common Information Model (CIM)
This standard is used by European TSO's 

## Problem description
Code that infers xml schema based on example files can result in "silent" errors, causing poor data quality.
When too small integer types are chosen this might cause missing data or wrong data with negative numbers.
When large integers are chosen where string should have been selected the column content can become empty, causing duplicates.
Further nested arrays might not be flattened correct, causing missing data!

Is there a way to **generate** the below Python Struct Type definition directly from provided XSD's (xml schema definition files)?

### Example

Example files and xsd's can be found here
https://gitlab.entsoe.eu/transparency/xml-examples/-/blob/main/Generation/Aggregated%20Generation%20Per%20Type%20%5B16.1.B&C%5D.xml?ref_type=heads

Renamed example file: Generation/Aggregated Generation Per Type [16.1.B&C].xml -> a16-1bc.xml

### Manual created Struct Type Definition
Inspired by Copilot, but which was not able to provide complete working code

In [0]:
%python

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType,
    DoubleType, DecimalType, ArrayType
)

xml_file = '/mnt/bd/openuniverselake/aisraw/entsoe_tp/my_examples/a16-1bc.xml'  # CONFIG!

# causes last 2 columns to become empty :-0
#interval_schema = StructType([
#    StructField("start", TimestampType(), False),
#    StructField("end", TimestampType(), False)
#])  

interval_schema = StructType([
    StructField("start", StringType(), False),
    StructField("end", StringType(), False)
])  

point_schema = StructType([
    StructField("position", IntegerType(), False),
    StructField("quantity", DecimalType(15, 6), True) 
])

period_schema = StructType([
    StructField("timeInterval", StructType(interval_schema), False),  # -> interval_schema
    StructField("resolution", StringType(), False),
    StructField("Point", ArrayType(point_schema), False)
])

timeseries_schema = StructType([
    StructField("mRID", StringType(), False),
    StructField("businessType", StringType(), False),
    StructField("objectAggregation", StringType(), False),
    StructField("inBiddingZone_Domain.mRID", StringType(), True),
    StructField("quantity_Measure_Unit.name", StringType(), True),
    StructField("curveType", StringType(), False),
    StructField("MktPSRType", StructType([StructField("psrType", StringType(), True)]), True),
    StructField("Period", ArrayType(period_schema), False)
])

gl_market_schema = StructType([
    StructField("mRID", StringType(), False),
    StructField("revisionNumber", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("process.processType", StringType(), False),
    StructField("sender_MarketParticipant.mRID", StringType(), False),
    StructField("sender_MarketParticipant.marketRole.type", StringType(), False),
    StructField("receiver_MarketParticipant.mRID", StringType(), False),
    StructField("receiver_MarketParticipant.marketRole.type", StringType(), False),
    StructField("createdDateTime", TimestampType(), False),
    StructField("time_Period.timeInterval", StructType(interval_schema), True),   # -> interval_schema
    StructField("TimeSeries", ArrayType(timeseries_schema), True)
])

df = spark.read.format("xml") \
    .option("rowTag", "GL_MarketDocument") \
    .schema(gl_market_schema) \
    .load(xml_file)

df.createOrReplaceTempView("gl_market")

### Example data

In [0]:
SELECT * 
FROM gl_market;

### Standardization 

In [0]:
CREATE OR REPLACE TEMPORARY VIEW new_data
AS
SELECT DATEADD(minute, (x.TimeSeries_Period_Point_position-1)*r.minutes, TimeSeries_Period_timeInterval_start ) as timeseries_timestamp
     ,  x.*
FROM (  SELECT   m.mRID 
               , m.revisionNumber             
               , m.type
               , m.`process.processType`                                                      as process_processType
               , m.`sender_MarketParticipant.mRID`                                            as sender_MarketParticipant_mRID
               , m.`sender_MarketParticipant.marketRole.type`                                 as sender_MarketParticipant_marketRole_type
               , m.`receiver_MarketParticipant.mRID`                                          as receiver_MarketParticipant_mRID
               , m.`receiver_MarketParticipant.marketRole.type`                               as receiver_MarketParticipant_marketRole_type
               , m.createdDateTime            
               , to_timestamp(replace(m.`time_Period.timeInterval`.start, ':00Z', ':00:00Z')) as time_Period_timeInterval_start
               , to_timestamp(replace(m.`time_Period.timeInterval`.end, ':00Z', ':00:00Z') )  as time_Period_timeInterval_end
               , ts.mRID                                                                      as TimeSeries_mRID
               , ts.businessType                                                              as TimeSeries_businessType
               , ts.objectAggregation                                                         as TimeSeries_objectAggregation
               , ts.`inBiddingZone_Domain.mRID`                                               as TimeSeries_inBiddingZone_Domain_mRID
               , ts.`quantity_Measure_Unit.name`                                              as TimeSeries_quantity_Measure_Unit_name
               , ts.curveType                                                                 as TimeSeries_curveType
               , ts.MktPSRType.psrType                                                        as TimeSeries_MktPSRType_psrType
               , to_timestamp(replace(p.timeInterval.start, ':00Z', ':00:00Z'))               as TimeSeries_Period_timeInterval_start    
               , to_timestamp(replace(p.timeInterval.end, ':00Z', ':00:00Z'))                 as TimeSeries_Period_timeInterval_end
               , p.resolution                                                                 as TimeSeries_Period_resolution
               , pt.position                                                                  as TimeSeries_Period_Point_position 
               , pt.quantity                                                                  as TimeSeries_Period_Point_quantity
          FROM gl_market m
          LATERAL VIEW explode(m.TimeSeries) ts_table AS ts
          LATERAL VIEW explode(ts.Period) p_table AS p
          LATERAL VIEW explode(p.Point) pt_table AS pt
     )    x
LEFT OUTER JOIN ( VALUES ('PT1H', 60)
                       , ('PT60M', 60)
                       , ('PT30M', 30)
                       , ('PT15M', 15)
                       , ('PT5M', 5) ) AS r(resolution, minutes) ON x.TimeSeries_Period_resolution = r.resolution
;

### Flattened result

In [0]:
SELECT * FROM new_data;